<img src="https://cros.ec.europa.eu/system/files/inline-images/logo%20official%20estp_0.jpg" height="120px">

# ADVANCED PYTHON FOR OFFICIAL STATISTICS
## ICON-Institute · Cologne · June 22–26, 2026 · Day 1 PM
### [Dr. Christian Kauth](https://www.linkedin.com/in/ckauth/)

---

# 📦 Notebook 02 — Packaging & Testing
## *Advanced Python for Official Statistics*

> *"Code without tests is broken code that hasn't been caught yet."*

This afternoon we turn the `silc_toolkit` code from this morning into a **professional, installable Python package** — complete with automated tests, a consistent code style, and a working debugger. By the end of this session every tool we build for the rest of the week will live in this package.

| Topic | You will be able to… |
|-------|---------------------|
| **Packages** | Structure `silc_toolkit` with `__init__.py`, sub-modules, and a proper `pyproject.toml` |
| **Type hints** | Annotate all public functions; use `Optional`, `Sequence`, `TypeVar` |
| **pytest** | Write fixtures, parametrised tests, and basic mocks for SILC functions |
| **Coverage** | Measure which lines are tested; identify gaps |
| **black / ruff** | Auto-format the codebase to PEP 8 in one command |
| **logging / pdb** | Debug production code without `print()` statements |

---
## 0 · Setup

In [ ]:
import os
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.toml not found")

PROJECT_ROOT = find_project_root(Path().resolve())
os.chdir(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
print(f"✅ Project root: {PROJECT_ROOT}")

✅ Project root: D:\Local\ICON\Python_Advanced\python_advanced


### `pyproject.toml` update — Notebook 02

This notebook adds: **polars>=1.40, pytest, black, ruff (dev tools)**

Run the cell below to update `pyproject.toml` and install the new packages.

In [ ]:
%%writefile pyproject.toml
# Day 1+2: adds testing tools and polars
# New in notebook 02:
#   + polars>=1.40
#   + pytest, black, ruff (dev tools)

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "silc-toolkit"
version = "0.1.0"
description = "Advanced Python for Official Statistics -- EU-SILC Analysis Toolkit"
requires-python = ">=3.11"
authors = [{ name = "Christian Kauth", email = "christian.kauth@unifr.ch" }]
license = { text = "MIT" }

dependencies = [
    "pydantic>=2.5",
    "pandas>=2.1",
    "pyarrow>=14.0",
    "polars>=1.40",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.4",
    "pytest-cov>=4.1",
    "black>=23.0",
    "ruff>=0.3",
]

[project.urls]
Repository = "https://github.com/icon-institute/silc-toolkit"

[tool.hatch.build.targets.wheel]
packages = ["silc_toolkit"]

[tool.hatch.build.targets.sdist]
# Keep the source distribution lean — the PUF data is not part of the package
exclude = ["data", "dist", "src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-v --tb=short"

[tool.black]
line-length = 88
target-version = ["py311"]

[tool.ruff]
line-length = 88
target-version = "py311"
extend-exclude = ["*.ipynb"]

[tool.ruff.lint]
select = ["E", "F", "I", "UP"]
ignore = ["E501"]

[tool.ruff.lint.isort]
known-first-party = ["silc_toolkit"]

[tool.ruff.format]
quote-style = "double"   # default; matches black — explicit for clarity
indent-style = "space"   # default

Overwriting pyproject.toml


In [ ]:
# Sync all dependencies declared in pyproject.toml
# Run this after every %%writefile pyproject.toml cell
import subprocess
result = subprocess.run(
    ["uv", "pip", "install", "-e", "."],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("silc-toolkit + notebook 02 deps installed")
else:
    print("uv error:", result.stderr[-500:])

silc-toolkit + notebook 02 deps installed


---
## 1 · Package Structure

### 1.1 What Makes a Python Package?

A **package** is a directory with an `__init__.py` file. Any `.py` files inside become **modules** that can be imported with dot notation.

```
silc_toolkit/             ← package directory
├── __init__.py           ← makes it a package; defines public API
├── models.py             ← Pydantic models (built in notebook 01)
├── functional.py         ← decorators, generators (notebook 01)
├── loaders.py            ← pandas / CSV loading helpers (today)
├── indicators.py         ← AROP, Gini, AROPE functions (today)
└── _version.py           ← single source of truth for version
```

The user installs with: `pip install -e .` (editable install — changes take effect immediately).

### 1.2 The `pyproject.toml`

Modern Python packaging uses `pyproject.toml` (PEP 518 / 621). Open ours:

In [ ]:
print((PROJECT_ROOT / "pyproject.toml").read_text(encoding="utf-8"))

# Day 1+2: adds testing tools and polars
# New in notebook 02:
#   + polars>=1.40
#   + pytest, black, ruff (dev tools)

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "silc-toolkit"
version = "0.1.0"
description = "Advanced Python for Official Statistics -- EU-SILC Analysis Toolkit"
requires-python = ">=3.11"
authors = [{ name = "Christian Kauth", email = "christian.kauth@unifr.ch" }]
license = { text = "MIT" }

dependencies = [
    "pydantic>=2.5",
    "pandas>=2.1",
    "pyarrow>=14.0",
    "polars>=1.40",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.4",
    "pytest-cov>=4.1",
    "black>=23.0",
    "ruff>=0.3",
]

[project.urls]
Repository = "https://github.com/icon-institute/silc-toolkit"

[tool.hatch.build.targets.wheel]
packages = ["silc_toolkit"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-v --tb=short"

[tool.black]
line-length = 88
target-version = ["py311"]

[tool.ruff]
line-length = 88
target-version

### 1.3 Editable Install

Running `pip install -e .` tells pip to link to our source directory rather than copying. Changes to `.py` files are immediately available without reinstalling.

In [ ]:
import subprocess
result = subprocess.run(
    ["uv", "pip", "install", "-e", ".", "--quiet"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ silc_toolkit installed in editable mode")
else:
    print("Installation output:", result.stdout[-500:] if result.stdout else "")
    print("Errors:", result.stderr[-500:] if result.stderr else "")

✅ silc_toolkit installed in editable mode


---
## 2 · Type Hints

Type hints make code **self-documenting** and enable static analysis tools (mypy, pyright) to catch bugs before runtime.

```python
# Without type hints — what does this return?
def compute_arop(incomes, threshold):
    ...

# With type hints — crystal clear
def compute_arop(incomes: list[float], threshold: float = 0.60) -> float:
    ...
```

Key types to know:

In [ ]:
from __future__ import annotations   # enables string-based forward references

from typing import Optional, Sequence, TypeVar, Callable, Iterator, Any
from collections.abc import Generator

T = TypeVar("T")   # generic type variable


# Type aliases improve readability for domain-specific types
IncomeList = list[float]
CountryCode = str          # e.g. "LU", "BE"
YearType    = int          # e.g. 2012


def load_incomes(
    country: CountryCode,
    year: YearType,
    data_dir: Path,
    *, # every parameter after this must be passed as a keyword argument
    min_income: float = 0.0,
    max_rows: Optional[int] = None,
) -> IncomeList:
    """
    Load equivalised disposable incomes from the H-file.

    Parameters
    ----------
    country   : ISO 2-letter code (e.g. 'LU', 'BE')
    year      : Survey wave year (2004–2023)
    data_dir  : Root directory containing country sub-folders
    min_income: Minimum income to include (default: 0 — exclude negatives)
    max_rows  : If set, stop after this many rows (useful for testing)

    Returns
    -------
    List of equivalised disposable incomes in euros
    """
    import csv
    path = data_dir / f"{country}_PUF_EUSILC" / f"{country}_{year}h_EUSILC.csv"
    if not path.exists():
        raise FileNotFoundError(f"No H-file for {country} {year}: {path}")

    incomes: IncomeList = []
    with open(path, newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        for i, row in enumerate(reader):
            if max_rows is not None and i >= max_rows:
                break
            inc_str  = row.get("HY020", "").strip()
            size_str = row.get("HX040", "").strip()
            if not inc_str or inc_str in ("NA", ""):
                continue
            income = float(inc_str)
            size   = max(1, int(float(size_str))) if size_str and size_str != "NA" else 1
            equiv  = income / (1.0 + 0.5 * (size - 1))
            if equiv >= min_income:
                incomes.append(equiv)
    return incomes


# Test it on a couple of participant countries
import pandas as pd
for country in ["BE", "ES", "HU", "IE"]:
    try:
        inc = load_incomes(country, 2012, DATA_DIR)
        print(f"  {country}: {len(inc):,} households loaded, "
              f"median = €{sorted(inc)[len(inc)//2]:,.0f}")
    except FileNotFoundError:
        print(f"  {country}: no data")

  BE: 5,395 households loaded, median = €20,455
  ES: 11,955 households loaded, median = €14,411
  HU: 11,062 households loaded, median = €7,523
  IE: 4,211 households loaded, median = €20,065


---
## 3 · Adding `loaders.py` and `indicators.py` to the Package

In [ ]:
%%writefile silc_toolkit/loaders.py
"""
CSV and pandas loaders for EU-SILC PUF data.

All functions accept a ``data_dir`` argument so they work with any copy
of the PUF, regardless of installation path.
"""
from __future__ import annotations

import csv
import logging
from pathlib import Path
from typing import Optional

import pandas as pd

logger = logging.getLogger(__name__)

# Columns we always want from the H-file
H_COLS_MINIMAL = ["HB010", "HB020", "HB030", "HY010", "HY020", "HX040"]
# Columns from the D (register) file — household weight and region
D_COLS_MINIMAL = ["DB010", "DB020", "DB030", "DB040", "DB090"]


def load_incomes(
    country: str,
    year: int,
    data_dir: Path,
    *,
    min_income: float = 0.0,
    max_rows: Optional[int] = None,
) -> list[float]:
    """
    Load equivalised disposable incomes from the H-file.
    Returns a list of positive equivalised incomes in euros.
    """
    path = data_dir / f"{country}_PUF_EUSILC" / f"{country}_{year}h_EUSILC.csv"
    if not path.exists():
        raise FileNotFoundError(f"No H-file for {country} {year}: {path}")

    incomes: list[float] = []
    with open(path, newline="", encoding="utf-8") as fh:
        for i, row in enumerate(csv.DictReader(fh)):
            if max_rows is not None and i >= max_rows:
                break
            inc_str  = row.get("HY020", "").strip()
            size_str = row.get("HX040", "").strip()
            if not inc_str or inc_str == "NA":
                continue
            income = float(inc_str)
            size   = max(1, int(float(size_str))) if size_str and size_str != "NA" else 1
            equiv  = income / (1.0 + 0.5 * (size - 1))
            if equiv >= min_income:
                incomes.append(equiv)
    return incomes


def load_household_df(
    country: str,
    year: int,
    data_dir: Path,
    cols: Optional[list[str]] = None,
) -> pd.DataFrame:
    """
    Load the H-file for one country-year into a pandas DataFrame.
    Merges with the D-file to add weight (DB090) and region (DB040).
    """
    cols = cols or H_COLS_MINIMAL
    h_path = data_dir / f"{country}_PUF_EUSILC" / f"{country}_{year}h_EUSILC.csv"
    d_path = data_dir / f"{country}_PUF_EUSILC" / f"{country}_{year}d_EUSILC.csv"

    if not h_path.exists():
        raise FileNotFoundError(f"H-file not found: {h_path}")

    # Read H-file; keep only requested columns that exist
    df_h = pd.read_csv(h_path)
    available_h = [c for c in cols if c in df_h.columns]
    df_h = df_h[available_h].copy()

    # Optionally merge D-file for weight + region
    if d_path.exists():
        df_d = pd.read_csv(d_path, usecols=lambda c: c in D_COLS_MINIMAL)
        if "DB030" in df_d.columns and "HB030" in df_h.columns:
            df = df_h.merge(
                df_d[["DB030", "DB090", "DB040"]],
                left_on="HB030", right_on="DB030",
                how="left",
            ).drop(columns="DB030", errors="ignore")
        else:
            df = df_h
    else:
        df = df_h

    # Type coercion
    for col in ["HY010", "HY020"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
    if "HX040" in df.columns:
        df["HX040"] = pd.to_numeric(df["HX040"], errors="coerce").fillna(1).astype(int)
    if "DB090" in df.columns:
        df["DB090"] = pd.to_numeric(df["DB090"], errors="coerce").fillna(1.0)

    # Derived column: equivalised income
    if "HY020" in df.columns and "HX040" in df.columns:
        df["equiv_income"] = df["HY020"] / (1.0 + 0.5 * (df["HX040"] - 1)).clip(lower=1)

    logger.info(
        "Loaded %s %d: %d households, %d columns",
        country, year, len(df), len(df.columns)
    )
    return df


def load_multi_year(
    country: str,
    years: list[int],
    data_dir: Path,
) -> pd.DataFrame:
    """Concatenate household DataFrames across multiple years for one country."""
    frames = []
    for year in years:
        try:
            df = load_household_df(country, year, data_dir)
            df["survey_year"] = year
            frames.append(df)
        except FileNotFoundError:
            logger.warning("No H-file for %s %d — skipped", country, year)
    if not frames:
        raise ValueError(f"No data found for {country} in years {years}")
    return pd.concat(frames, ignore_index=True)

Overwriting silc_toolkit/loaders.py


In [ ]:
%%writefile silc_toolkit/indicators.py
"""
EU-SILC poverty and inequality indicators.

All functions take plain ``list[float]`` or numpy arrays for testability.
Weighted versions accept a parallel list of weights.
"""
from __future__ import annotations

import math
import statistics
from typing import Optional, Sequence


def median_income(incomes: Sequence[float]) -> float:
    """Median of a sequence of incomes."""
    pos = [x for x in incomes if x > 0]
    if not pos:
        return 0.0
    return statistics.median(pos)


def poverty_threshold(
    incomes: Sequence[float],
    threshold_pct: float = 0.60,
) -> float:
    """
    EU at-risk-of-poverty threshold.
    Default: 60% of median equivalised disposable income (EU definition).
    """
    return threshold_pct * median_income(incomes)


def arop_rate(
    incomes: Sequence[float],
    threshold_pct: float = 0.60,
    weights: Optional[Sequence[float]] = None,
) -> float:
    """
    At-risk-of-poverty rate (AROP).

    Parameters
    ----------
    incomes       : equivalised disposable household incomes
    threshold_pct : fraction of median (default 0.60 = EU definition)
    weights       : household design weights (DB090); if None, unweighted

    Returns
    -------
    Rate as a fraction in [0, 1]
    """
    if not incomes:
        return 0.0
    thresh = poverty_threshold(incomes, threshold_pct)
    if weights is None:
        at_risk = sum(1 for i in incomes if i < thresh)
        return at_risk / len(incomes)
    else:
        w_poor  = sum(w for i, w in zip(incomes, weights) if i < thresh)
        w_total = sum(weights)
        return w_poor / w_total if w_total > 0 else 0.0


def gini_coefficient(incomes: Sequence[float]) -> float:
    """
    Gini coefficient of income inequality.
    Range: 0 (perfect equality) to 1 (maximum inequality).
    Uses the efficient sorted-rank formula.
    """
    xs = sorted(x for x in incomes if x > 0)
    n  = len(xs)
    if n == 0:
        return 0.0
    total = sum(xs)
    if total == 0:
        return 0.0
    cumsum = sum(x * (2 * rank - n - 1) for rank, x in enumerate(xs, 1))
    return cumsum / (n * total)


def material_deprivation_rate(flags: Sequence[int], threshold: int = 3) -> float:
    """
    Severe material deprivation rate.
    A household is severely deprived if it is unable to afford
    at least ``threshold`` items from the standard EU-SILC list.

    Parameters
    ----------
    flags     : number of deprivation items per household (0–9)
    threshold : minimum items lacking to be classified as deprived (default 3)
    """
    if not flags:
        return 0.0
    deprived = sum(1 for f in flags if f >= threshold)
    return deprived / len(flags)


def s80s20_ratio(incomes: Sequence[float]) -> float:
    """
    Income quintile share ratio (S80/S20).
    Total income of top 20% divided by total income of bottom 20%.
    """
    xs = sorted(x for x in incomes if x > 0)
    n  = len(xs)
    if n < 5:
        return float("nan")
    q = n // 5
    bottom_20 = sum(xs[:q])
    top_20    = sum(xs[-q:])
    return top_20 / bottom_20 if bottom_20 > 0 else float("inf")

Overwriting silc_toolkit/indicators.py


In [ ]:
%%writefile silc_toolkit/__init__.py
"""
silc_toolkit — EU-SILC Analysis Toolkit
Advanced Python for Official Statistics · ICON-Institute 2026
"""
from .models import HouseholdModel, PersonModel, SurveyWave
from .functional import timer, log_call, validate_output, retry
from .loaders import load_incomes, load_household_df, load_multi_year
from .indicators import arop_rate, gini_coefficient, poverty_threshold, s80s20_ratio

__version__ = "0.1.0"
__all__ = [
    # models
    "HouseholdModel", "PersonModel", "SurveyWave",
    # functional
    "timer", "log_call", "validate_output", "retry",
    # loaders
    "load_incomes", "load_household_df", "load_multi_year",
    # indicators
    "arop_rate", "gini_coefficient", "poverty_threshold", "s80s20_ratio",
]

Overwriting silc_toolkit/__init__.py


In [ ]:
# Reload and smoke-test the full package
import importlib, sys
for mod in list(sys.modules):
    if mod.startswith("silc_toolkit"):
        del sys.modules[mod]

from silc_toolkit import (
    load_incomes, load_household_df,
    arop_rate, gini_coefficient, s80s20_ratio,
)

# Test on Ireland 2012
inc_ie = load_incomes("IE", 2012, DATA_DIR)
print(f"Ireland 2012:")
print(f"  Households    : {len(inc_ie):,}")
print(f"  AROP rate     : {arop_rate(inc_ie):.1%}")
print(f"  Gini          : {gini_coefficient(inc_ie):.4f}")
print(f"  S80/S20 ratio : {s80s20_ratio(inc_ie):.2f}")

# Test on Greece 2012 (uses 'EL' country code in SILC)
inc_el = load_incomes("EL", 2012, DATA_DIR)
print(f"\nGreece 2012:")
print(f"  AROP rate     : {arop_rate(inc_el):.1%}")
print(f"  Gini          : {gini_coefficient(inc_el):.4f}")

Ireland 2012:
  Households    : 4,211
  AROP rate     : 26.7%
  Gini          : 0.4514
  S80/S20 ratio : 13.13

Greece 2012:
  AROP rate     : 26.1%
  Gini          : 0.4056


---
## 4 · Testing with pytest

### Why Test?

- Gini coefficients must be in [0, 1]
- AROP rate must be in [0, 1]
- Loading an empty file must not crash silently
- A refactored Gini function must give the same result as the old one

Without tests, we discover these failures in **production** — when the wrong poverty rate ends up in a ministerial report.

### pytest Building Blocks

| Feature | Purpose |
|---------|---------|
| `def test_*()` | Any function starting with `test_` is a test |
| `assert expr` | Fails the test if `expr` is falsy |
| `@pytest.fixture` | Shared setup code (test data, DB connections) |
| `@pytest.mark.parametrize` | Run the same test with many inputs |
| `pytest.raises(Exc)` | Assert that code raises a specific exception |

In [ ]:
import os
os.makedirs("tests", exist_ok=True)
print("✅ tests/ directory ready")

✅ tests/ directory ready


In [ ]:
%%writefile tests/conftest.py
"""
pytest configuration and shared fixtures for silc_toolkit tests.
"""
from __future__ import annotations

from pathlib import Path

import pytest

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

@pytest.fixture(scope="session")
def data_dir() -> Path:
    """Absolute path to the project data/ directory."""
    root = Path(__file__).parent.parent
    d = root / "data"
    if not d.exists():
        pytest.skip("data/ directory not found — skipping data-dependent tests")
    return d


@pytest.fixture(scope="session")
def lu_incomes(data_dir) -> list[float]:
    """Luxembourg 2012 equivalised incomes (loaded once per session)."""
    from silc_toolkit.loaders import load_incomes
    return load_incomes("LU", 2012, data_dir)


@pytest.fixture(scope="session")
def ie_incomes(data_dir) -> list[float]:
    """Ireland 2012 equivalised incomes."""
    from silc_toolkit.loaders import load_incomes
    return load_incomes("IE", 2012, data_dir)


# ---------------------------------------------------------------------------
# Small synthetic income lists for fast unit tests (no file I/O)
# ---------------------------------------------------------------------------

@pytest.fixture
def perfect_equality() -> list[float]:
    """50 identical incomes → Gini = 0."""
    return [20_000.0] * 50


@pytest.fixture
def perfect_inequality() -> list[float]:
    """49 zeros + 1 large income → Gini ≈ 1."""
    return [0.0] * 49 + [1_000_000.0]


@pytest.fixture
def small_wave() -> list[float]:
    """
    10 synthetic household incomes for readable expected values.
    Median = (8_000 + 10_000) / 2 = 9_000
    60% threshold = 5_400
    At-risk households: 3_000, 4_000, 5_000 → AROP = 3/10 = 0.30
    """
    return [3_000, 4_000, 5_000, 6_000, 8_000,
            10_000, 12_000, 15_000, 20_000, 30_000]

Overwriting tests/conftest.py


In [ ]:
%%writefile tests/test_indicators.py
"""
Tests for silc_toolkit.indicators — AROP, Gini, S80/S20.
"""
from __future__ import annotations

import math

import pytest

from silc_toolkit.indicators import (
    arop_rate,
    gini_coefficient,
    median_income,
    poverty_threshold,
    s80s20_ratio,
)


# ---------------------------------------------------------------------------
# median_income
# ---------------------------------------------------------------------------

def test_median_income_basic(small_wave):
    assert median_income(small_wave) == 9_000.0


def test_median_income_empty():
    assert median_income([]) == 0.0


def test_median_income_ignores_non_positive():
    # Negative incomes (self-employment losses) must be excluded from median
    inc = [-5_000, 0, 10_000, 20_000]
    assert median_income(inc) == pytest.approx(15_000.0)


# ---------------------------------------------------------------------------
# poverty_threshold
# ---------------------------------------------------------------------------

def test_poverty_threshold_default(small_wave):
    assert poverty_threshold(small_wave) == pytest.approx(5_400.0)


@pytest.mark.parametrize("pct, expected", [
    (0.50, 4_500.0),
    (0.60, 5_400.0),
    (0.70, 6_300.0),
])
def test_poverty_threshold_pct(small_wave, pct, expected):
    assert poverty_threshold(small_wave, pct) == pytest.approx(expected)


# ---------------------------------------------------------------------------
# arop_rate
# ---------------------------------------------------------------------------

def test_arop_rate_basic(small_wave):
    # 3 households (3_000, 4_000, 5_000) below threshold of 5_400
    assert arop_rate(small_wave) == pytest.approx(0.30)


def test_arop_rate_empty():
    assert arop_rate([]) == 0.0


def test_arop_rate_all_rich():
    inc = [100_000.0] * 10
    assert arop_rate(inc) == 0.0


def test_arop_rate_all_poor():
    # All below any realistic threshold
    inc = [100.0] * 10
    assert arop_rate(inc) == 0.0  # median = 100, threshold = 60, all > 60


def test_arop_rate_weighted(small_wave):
    # Households 1-3 (at risk) have weight 2; others weight 1
    weights = [2.0, 2.0, 2.0] + [1.0] * 7
    # Weighted at-risk: 2+2+2 = 6; total weight = 6+7 = 13
    result = arop_rate(small_wave, weights=weights)
    assert result == pytest.approx(6 / 13)


@pytest.mark.parametrize("country_label, expected_range", [
    ("LU", (0.05, 0.30)),
    ("IE", (0.10, 0.30)),
])
def test_arop_rate_realistic_range(
    country_label,
    expected_range,
    request #built-in pytest fixture
    ):
    """Integration test: AROP must be in a plausible range for each country."""
    incomes = request.getfixturevalue(f"{country_label.lower()}_incomes")
    rate = arop_rate(incomes)
    lo, hi = expected_range
    assert lo <= rate <= hi, (
        f"{country_label} AROP = {rate:.3f}, expected [{lo}, {hi}]"
    )


# ---------------------------------------------------------------------------
# gini_coefficient
# ---------------------------------------------------------------------------

def test_gini_perfect_equality(perfect_equality):
    assert gini_coefficient(perfect_equality) == pytest.approx(0.0)


def test_gini_perfect_inequality(perfect_inequality):
    # 49 zeros filtered out; only 1 positive value → undefined or 0
    # Function filters zeros, so only one element → Gini = 0
    g = gini_coefficient(perfect_inequality)
    assert 0.0 <= g <= 1.0


def test_gini_empty():
    assert gini_coefficient([]) == 0.0


def test_gini_in_range(lu_incomes):
    g = gini_coefficient(lu_incomes)
    assert 0.0 <= g <= 1.0


def test_gini_known_value():
    # Simple 4-person economy: incomes [1, 2, 3, 4]
    # Gini = (1*(2*1-4-1) + 2*(2*2-4-1) + 3*(2*3-4-1) + 4*(2*4-4-1)) / (4*10)
    # = (1*(-3) + 2*(-1) + 3*(1) + 4*(3)) / 40
    # = (-3 - 2 + 3 + 12) / 40 = 10/40 = 0.25
    assert gini_coefficient([1.0, 2.0, 3.0, 4.0]) == pytest.approx(0.25)


# ---------------------------------------------------------------------------
# s80s20_ratio
# ---------------------------------------------------------------------------

def test_s80s20_perfect_equality():
    # All equal → ratio = 1.0
    inc = [10_000.0] * 100
    assert s80s20_ratio(inc) == pytest.approx(1.0)


def test_s80s20_empty():
    assert math.isnan(s80s20_ratio([1.0, 2.0]))


def test_s80s20_realistic(lu_incomes):
    ratio = s80s20_ratio(lu_incomes)
    # Luxembourg is a wealthy country; ratio should be 3–8
    assert 2.0 <= ratio <= 15.0

Overwriting tests/test_indicators.py


In [ ]:
%%writefile tests/test_loaders.py
"""
Tests for silc_toolkit.loaders.
"""
from __future__ import annotations

from pathlib import Path

import pytest

from silc_toolkit.loaders import load_incomes, load_household_df


def test_load_incomes_returns_list(data_dir):
    inc = load_incomes("LU", 2012, data_dir)
    assert isinstance(inc, list)
    assert all(isinstance(x, float) for x in inc)


def test_load_incomes_positive_only(data_dir):
    inc = load_incomes("LU", 2012, data_dir, min_income=0.0)
    assert all(x >= 0.0 for x in inc), "min_income=0 should exclude negatives"


def test_load_incomes_missing_country(data_dir):
    with pytest.raises(FileNotFoundError, match="No H-file"):
        load_incomes("XX", 2012, data_dir)


def test_load_incomes_max_rows(data_dir):
    inc = load_incomes("BE", 2012, data_dir, max_rows=10)
    assert len(inc) <= 10


def test_load_household_df_shape(data_dir):
    df = load_household_df("LU", 2012, data_dir)
    assert len(df) > 0
    assert "HY020" in df.columns
    assert "equiv_income" in df.columns


def test_load_household_df_equiv_income_non_negative(data_dir):
    df = load_household_df("LU", 2012, data_dir)
    # equiv_income may be negative (losses) — just check no NaN
    assert df["equiv_income"].isna().sum() == 0


@pytest.mark.parametrize("country", ["BE", "ES", "HU", "IE", "SE"])
def test_load_incomes_participant_countries(data_dir, country):
    """All participant countries must load without error."""
    inc = load_incomes(country, 2012, data_dir)
    assert len(inc) > 0, f"No incomes loaded for {country}"

Overwriting tests/test_loaders.py


In [ ]:
# Install dev dependencies for testing and linting
import subprocess
result = subprocess.run(
    ["uv", "pip", "install", "-e", ".[dev]"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("silc-toolkit + notebook 02 deps installed, including dev dependencies")
else:
    print("uv error:", result.stderr[-500:])

silc-toolkit + notebook 02 deps installed, including dev dependencies


The `.[dev]` syntax tells `uv/pip` to install the package plus the dev optional dependencies group — which includes `pytest`, `pytest-cov`, `black`, and `ruff`.

In [ ]:
# Run the test suite
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short", "--no-header"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-500:])

============================= test session starts =============================
collecting ... collected 33 items

tests/test_indicators.py::test_median_income_basic PASSED                [  3%]
tests/test_indicators.py::test_median_income_empty PASSED                [  6%]
tests/test_indicators.py::test_median_income_ignores_non_positive PASSED [  9%]
tests/test_indicators.py::test_poverty_threshold_default PASSED          [ 12%]
tests/test_indicators.py::test_poverty_threshold_pct[0.5-4500.0] PASSED  [ 15%]
tests/test_indicators.py::test_poverty_threshold_pct[0.6-5400.0] PASSED  [ 18%]
tests/test_indicators.py::test_poverty_threshold_pct[0.7-6300.0] PASSED  [ 21%]
tests/test_indicators.py::test_arop_rate_basic PASSED                    [ 24%]
tests/test_indicators.py::test_arop_rate_empty PASSED                    [ 27%]
tests/test_indicators.py::test_arop_rate_all_rich PASSED                 [ 30%]
tests/test_indicators.py::test_arop_rate_all_poor PASSED                 [ 33%]
tests

### Pytest Fixture Scopes


Pytest fixture scopes control **how often** a fixture is created and torn down:

| Scope | Lifetime | Typical use case |
|-------|----------|-----------------|
| `function` *(default)* | Per test function | Mutable state that must be isolated between tests |
| `class` | Per test class | Shared setup for all methods in one `TestCase` |
| `module` | Per `.py` file | Expensive setup shared across all tests in a file |
| `package` | Per package directory | Shared across all tests in a package |
| `session` | Entire test run | Very expensive one-time setup (DB connection, loading big files) |

```python
@pytest.fixture(scope="function")   # default — new list for every test
def fresh_data():
    return []

@pytest.fixture(scope="module")     # CSV loaded once per test file
def loaded_dataframe():
    return pd.read_csv("big_file.csv")

@pytest.fixture(scope="session")    # connection opened once for the whole run
def db_connection():
    conn = create_connection()
    yield conn          # code after yield = teardown
    conn.close()
```

**Key rules**
- A fixture can only depend on fixtures of the **same or broader** scope  
  (e.g. a `module`-scoped fixture cannot use a `function`-scoped one).
- Use `yield` instead of `return` when you need **teardown** logic.
- In this course, `scope="session"` makes sense for EU-SILC data — the CSV/Parquet files are large and read-only.


---
## 5 · Code Coverage

Code coverage tells us which lines our tests actually execute. A line with 0% coverage is a potential bug that no test would catch.

In [ ]:
# Run with coverage
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/",
     "--cov=silc_toolkit", "--cov-report=term-missing",
     "--no-header", "-q"],
    capture_output=True, text=True,
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)

============================= test session starts =============================
collected 33 items

tests\test_indicators.py ......................                          [ 66%]
tests\test_loaders.py ...........                                        [100%]

=============================== tests coverage ================================
_______________ coverage: platform win32, python 3.13.9-final-0 _______________

Name                         Stmts   Miss  Cover   Missing
----------------------------------------------------------
silc_toolkit\__init__.py         6      0   100%
silc_toolkit\database.py        28     28     0%   5-56
silc_toolkit\functional.py      87     70    20%   24-32, 38-46, 52-65, 71-85, 95-101, 108-114, 121-130, 138-142, 148-151
silc_toolkit\indicators.py      45      5    89%   75, 91-94
silc_toolkit\loaders.py         66     15    77%   48, 72, 89-91, 119-129
silc_toolkit\mcp_server.py      70     70     0%   1-168
silc_toolkit\models.py          94     30

### VS Code: Viewing Test Coverage in the Editor

Install the [**Coverage Gutters**](https://marketplace.visualstudio.com/items?itemName=ryanluker.vscode-coverage-gutters) extension by ryanluker from the VS Code Extensions panel.

#### Generate a coverage file

Add `--cov-report=lcov:lcov.info` to your pytest call so the file is written with the exact filename Coverage Gutters expects.

This writes `lcov.info` to the project root.

#### Display coverage in the editor

Click **Watch** in the VS Code status bar (bottom left), or open the Command Palette (`Ctrl+Shift+P`) and run: `Coverage Gutters: Display Coverage`

In [ ]:
# Run with coverage
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/",
     "--cov=silc_toolkit", "--cov-report=term-missing", "--cov-report=lcov:lcov.info",
     "--no-header", "-q"],
    capture_output=True, text=True,
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)

============================= test session starts =============================
collected 33 items

tests\test_indicators.py ......................                          [ 66%]
tests\test_loaders.py ...........                                        [100%]

=============================== tests coverage ================================
_______________ coverage: platform win32, python 3.13.9-final-0 _______________

Name                         Stmts   Miss  Cover   Missing
----------------------------------------------------------
silc_toolkit\__init__.py         6      0   100%
silc_toolkit\database.py        28     28     0%   5-56
silc_toolkit\functional.py      87     70    20%   24-32, 38-46, 52-65, 71-85, 95-101, 108-114, 121-130, 138-142, 148-151
silc_toolkit\indicators.py      45      5    89%   75, 91-94
silc_toolkit\loaders.py         66     15    77%   48, 72, 89-91, 119-129
silc_toolkit\mcp_server.py      70     70     0%   1-168
silc_toolkit\models.py          94     30

---
## 6 · Code Quality with ruff (and black)

Modern Python projects use **ruff** for both linting *and* formatting in a single tool — one dependency, one config block, and 10–100× faster than the classic combination.

**black** was (and still is) widely used as a deterministic, zero-config formatter. We show a quick demo so you recognise it in existing projects.

**ruff** (written in Rust) replaces both:

| Command | What it does |
|---------|-------------|
| `ruff check` | **Linting** — unused imports (`F401`), undefined names (`F821`), style violations (`E*`), import order (`I*`), upgrade hints (`UP*`) |
| `ruff format` | **Formatting** — black-compatible auto-formatter; replaces `black` |

Both commands read their configuration from `pyproject.toml` under `[tool.ruff.lint]` and `[tool.ruff.format]`.

In [ ]:
# black: the classic formatter — still found in many codebases
# --check + --diff = dry run; shows what it WOULD change without touching files
result = subprocess.run(
    [sys.executable, "-m", "black", "--check", "--diff", "silc_toolkit/"],
    capture_output=True, text=True, encoding="utf-8"
)
print("black --check (dry run):")
if result.returncode == 0:
    print("  ✅ All files already conform to black style")
else:
    diff = (result.stdout or "")[:1000] or (result.stderr or "")[:1000]
    print(diff)

black --check (dry run):
--- D:\Local\ICON\Python_Advanced\python_advanced\silc_toolkit\__init__.py	2026-06-18 14:36:42.760967+00:00
+++ D:\Local\ICON\Python_Advanced\python_advanced\silc_toolkit\__init__.py	2026-06-18 14:37:00.217199+00:00
@@ -1,20 +1,31 @@
 """
 silc_toolkit — EU-SILC Analysis Toolkit
 Advanced Python for Official Statistics · ICON-Institute 2026
 """
+
 from .models import HouseholdModel, PersonModel, SurveyWave
 from .functional import timer, log_call, validate_output, retry
 from .loaders import load_incomes, load_household_df, load_multi_year
 from .indicators import arop_rate, gini_coefficient, poverty_threshold, s80s20_ratio
 
 __version__ = "0.1.0"
 __all__ = [
     # models
-    "HouseholdModel", "PersonModel", "SurveyWave",
+    "HouseholdModel",
+    "PersonModel",
+    "SurveyWave",
     # functional
-    "timer", "log_call", "validate_output", "retry",
+    "timer",
+    "log_call",
+    "validate_output",
+    "retry",
     # loaders
-    "load_incomes",

In [ ]:
# ruff check: linting — catches errors, style issues, unused imports, import order, etc.
result = subprocess.run(
    [sys.executable, "-m", "ruff", "check", "silc_toolkit/"],
    capture_output=True, text=True,
)
print("ruff check (linting):")
if result.returncode == 0:
    print("  ✅ No issues found")
else:
    print(result.stdout[:2000] or result.stderr[:2000])

ruff check (linting):
I001 [*] Import block is un-sorted or un-formatted
  --> silc_toolkit\__init__.py:5:1
   |
 3 |   Advanced Python for Official Statistics Â· ICON-Institute 2026
 4 |   """
 5 | / from .models import HouseholdModel, PersonModel, SurveyWave
 6 | | from .functional import timer, log_call, validate_output, retry
 7 | | from .loaders import load_incomes, load_household_df, load_multi_year
 8 | | from .indicators import arop_rate, gini_coefficient, poverty_threshold, s80s20_ratio
   | |____________________________________________________________________________________^
 9 |
10 |   __version__ = "0.1.0"
   |
help: Organize imports

UP035 `typing.List` is deprecated, use `list` instead
  --> silc_toolkit\database.py:8:1
   |
 7 | from pathlib import Path
 8 | from typing import List, Optional
   | ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 9 |
10 | from sqlalchemy import Float, ForeignKey, Integer, String, create_engine, text
   |

F401 [*] `sqlalchemy.text` imported but unused


In [ ]:
# ruff format: black-compatible formatter — dry run to preview changes
result = subprocess.run(
    [sys.executable, "-m", "ruff", "format", "--check", "--diff", "silc_toolkit/"],
    capture_output=True, text=True, encoding="utf-8"
)
print("ruff format --check (dry run):")
if result.returncode == 0:
    print("  ✅ All files already conform to ruff style")
else:
    diff = (result.stdout or "")[:1000] or (result.stderr or "")[:1000]
    print(diff)

ruff format --check (dry run):
--- silc_toolkit\__init__.py
+++ silc_toolkit\__init__.py
@@ -2,6 +2,7 @@
 silc_toolkit — EU-SILC Analysis Toolkit
 Advanced Python for Official Statistics · ICON-Institute 2026
 """
+
 from .models import HouseholdModel, PersonModel, SurveyWave
 from .functional import timer, log_call, validate_output, retry
 from .loaders import load_incomes, load_household_df, load_multi_year
@@ -10,11 +11,21 @@
 __version__ = "0.1.0"
 __all__ = [
     # models
-    "HouseholdModel", "PersonModel", "SurveyWave",
+    "HouseholdModel",
+    "PersonModel",
+    "SurveyWave",
     # functional
-    "timer", "log_call", "validate_output", "retry",
+    "timer",
+    "log_call",
+    "validate_output",
+    "retry",
     # loaders
-    "load_incomes", "load_household_df", "load_multi_year",



In [ ]:
# Apply ruff format — rewrites files in place (black-compatible output)
result = subprocess.run(
    [sys.executable, "-m", "ruff", "format", "silc_toolkit/", "tests/"],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout or result.stderr)
print("✅ ruff format complete")

6 files reformatted, 4 files left unchanged

✅ ruff format complete


### 6.1 Ruff in VS Code — Squiggly Lines Without Running a Script

The **Ruff** extension (`charliermarsh.ruff`) by Astral brings the same linting and formatting directly into the editor — no terminal command needed.

Once installed:
- **Linting errors** appear as squiggly underlines as you type (like flake8, but faster)
- Reads `[tool.ruff.lint]` config from `pyproject.toml` automatically
- Can act as the **format-on-save** provider, replacing black

#### Enable format-on-save

Open VS Code settings with `Ctrl+Shift+P` → *Open User Settings (JSON)* and add:

```json
"[python]": {
    "editor.formatOnSave": true,
    "editor.defaultFormatter": "charliermarsh.ruff"
}
```

Every `Ctrl+S` now runs `ruff format` silently. The subprocess cells above are useful for **CI pipelines and pre-commit hooks** — the extension handles the same job interactively during development.

> **Note:** Pylance (the VS Code Python language server) handles type checking and IntelliSense separately. Ruff and Pylance complement each other and are both active at the same time.

---
## 7 · Logging and Debugging

### 7.1 Structured Logging

`print()` is fine for exploration. Production code uses `logging` — it gives timestamps, severity levels, and can be redirected to files or monitoring systems.

### Logging Levels

Python's logging module defines five standard levels, in increasing severity:

| Level | When to use |
|-------|-------------|
| `DEBUG` | Detailed diagnostic info — values, loop counts, intermediate results. Only relevant when chasing a specific bug. |
| `INFO` | Confirmation that things are working as expected — "loaded 4 321 rows", "computation complete". |
| `WARNING` | Something unexpected happened, but the program can continue — missing optional file, fallback value used. |
| `ERROR` | A serious problem — an operation failed, a required resource is unavailable. Program may continue in a degraded state. |
| `CRITICAL` | A fatal error — the program cannot continue at all. |

**Rule of thumb for SILC work:**
- Use `DEBUG` for per-row or per-household detail (normally off in production)
- Use `INFO` for per-country or per-year milestones
- Use `WARNING` for missing country files (expected to happen occasionally)
- Use `ERROR` / `CRITICAL` for unexpected exceptions

The `basicConfig(level=logging.DEBUG)` call sets the **minimum** level that is shown. Change it to `logging.INFO` to silence all debug output in production.

#### Log record format attributes

The `format=` string uses `%`-style placeholders. Full reference:
[https://docs.python.org/3/library/logging.html#logrecord-attributes](https://docs.python.org/3/library/logging.html#logrecord-attributes)

Common attributes:

| Placeholder | Meaning |
|-------------|---------|
| `%(asctime)s` | Human-readable timestamp (`datefmt` controls the format) |
| `%(name)s` | Logger name — typically the module (`silc_toolkit.loaders`) |
| `%(levelname)s` | `DEBUG`, `INFO`, `WARNING`, `ERROR`, `CRITICAL` |
| `%(message)s` | The formatted log message |
| `%(filename)s` | Source file name |
| `%(lineno)d` | Line number in source file |

In [ ]:
import logging

LOG_FORMAT = "%(asctime)s  %(name)-20s  %(levelname)-8s  %(message)s"

# Configure root logger for this notebook
logging.basicConfig(
    level=logging.DEBUG,
    format=LOG_FORMAT,
    datefmt="%H:%M:%S",
)

# Uncomment to also write all log output to a file:
# _fh = logging.FileHandler("silc_toolkit.log", encoding="utf-8")
# _fh.setFormatter(logging.Formatter(LOG_FORMAT, datefmt="%H:%M:%S"))
# logging.getLogger().addHandler(_fh)

logger = logging.getLogger("silc_toolkit.demo")


def compute_country_indicators(country: str, year: int) -> dict:
    """Compute AROP and Gini for one country-year with structured logging."""
    logger.info("Starting computation for %s %d", country, year)

    try:
        from silc_toolkit import load_incomes, arop_rate, gini_coefficient
        incomes = load_incomes(country, year, DATA_DIR)
        logger.debug(f"Loaded {len(incomes)} incomes for {country} {year}")

        result = {
            "country":  country,
            "year":     year,
            "n":        len(incomes),
            "arop":     round(arop_rate(incomes), 4),
            "gini":     round(gini_coefficient(incomes), 4),
        }
        logger.info(
            "Completed %s %d: AROP=%.1f%%, Gini=%.4f",
            country, year, result["arop"]*100, result["gini"]
        )
        return result

    except FileNotFoundError:
        logger.warning("No PUF data for %s %d — skipping", country, year)
        return {"country": country, "year": year, "n": 0, "arop": None, "gini": None}
    except Exception as exc:
        logger.error("Unexpected error for %s %d: %s", country, year, exc, exc_info=True)
        raise


# Run for several participant countries
for country in ["BE", "FR", "IE", "SE", "MD"]:
    r = compute_country_indicators(country, 2012)
    if r["n"] > 0:
        print(f"  {r['country']}: AROP={r['arop']:.1%}  Gini={r['gini']:.4f}")

16:37:06  silc_toolkit.demo     INFO      Starting computation for BE 2012


16:37:06  silc_toolkit.demo     DEBUG     Loaded 5395 incomes for BE 2012
16:37:06  silc_toolkit.demo     INFO      Completed BE 2012: AROP=25.0%, Gini=0.3875
16:37:06  silc_toolkit.demo     INFO      Starting computation for FR 2012


  BE: AROP=25.0%  Gini=0.3875


16:37:06  silc_toolkit.demo     DEBUG     Loaded 11617 incomes for FR 2012
16:37:06  silc_toolkit.demo     INFO      Completed FR 2012: AROP=26.6%, Gini=0.4205
16:37:06  silc_toolkit.demo     INFO      Starting computation for IE 2012
16:37:06  silc_toolkit.demo     DEBUG     Loaded 4211 incomes for IE 2012
16:37:06  silc_toolkit.demo     INFO      Completed IE 2012: AROP=26.7%, Gini=0.4514
16:37:06  silc_toolkit.demo     INFO      Starting computation for SE 2012
16:37:06  silc_toolkit.demo     DEBUG     Loaded 6308 incomes for SE 2012
16:37:06  silc_toolkit.demo     INFO      Completed SE 2012: AROP=23.9%, Gini=0.3674


  FR: AROP=26.6%  Gini=0.4205
  IE: AROP=26.7%  Gini=0.4514
  SE: AROP=23.9%  Gini=0.3674


### 7.2 Debugging a `.py` Script in VS Code

For scripts and modules, VS Code's **Python Debugger** is far more productive seeding `print()`statements all over the place. It lets you pause on any line, inspect every variable, and step through logic — without touching the source code.

#### Step 1 — Write a script to debug

Run the cell below to write `debug_demo.py` to the project root.

#### Step 2 — Open the file and set a breakpoint

Open `debug_demo.py` in the editor. Click in the **gutter** (the narrow strip to the left of the line numbers) next to any line — a red dot appears. That is a **breakpoint**: execution will pause there before that line runs.

#### Step 3 — Launch the debugger

Click the **dropdown arrow** next to the ▷ Run button (top-right corner of the editor) and choose **Debug Python File** — or press `F5`.

#### Key debugger controls

| Action | Key | What it does |
|--------|-----|--------------|
| Continue | `F5` | Run until the next breakpoint |
| Step Over | `F10` | Execute the current line; stay in the same function |
| Step Into | `F11` | Jump inside the function being called |
| Step Out | `Shift+F11` | Finish the current function and return to the caller |
| Stop | `Shift+F5` | Terminate the debug session |

While paused, the **Variables** panel (left sidebar, *Run and Debug* view) shows every local and global variable at that exact moment. Hover over any name in the editor to see its value as an inline tooltip.

In [ ]:
import os
os.makedirs(PROJECT_ROOT / "src" / "scripts", exist_ok=True)
print("✅ src/scripts/ directory ready")

In [ ]:
%%writefile src/scripts/debug_demo.py
"""
debug_demo.py — a small script to step through in the VS Code Python Debugger.

Suggested breakpoints (click the gutter):
  • Line with  thresh = poverty_threshold(incomes)   inside summarise()
  • Line with  flagged.append(r["label"])             inside flag_high_poverty()
"""
from silc_toolkit.indicators import arop_rate, gini_coefficient, poverty_threshold


def summarise(label: str, incomes: list[float]) -> dict:
    thresh = poverty_threshold(incomes)          # ← breakpoint here
    rate   = arop_rate(incomes)
    gini   = gini_coefficient(incomes)
    return {"label": label, "arop": rate, "gini": gini, "n": len(incomes), "thresh": thresh}


def flag_high_poverty(results: list[dict], threshold: float = 0.20) -> list[str]:
    """Return labels where AROP exceeds the threshold."""
    flagged = []
    for r in results:
        if r["arop"] > threshold:
            flagged.append(r["label"])           # ← breakpoint here
    return flagged


if __name__ == "__main__":
    waves = {
        "Wave A": [3_000,  5_000,  8_000, 12_000, 20_000] * 20,
        "Wave B": [8_000, 10_000, 14_000, 18_000, 25_000] * 20,
        "Wave C": [2_000,  3_000,  4_000,  6_000,  9_000] * 20,
    }

    results = [summarise(label, inc) for label, inc in waves.items()]

    for r in results:
        print(f"{r['label']}: AROP={r['arop']:.1%}  Gini={r['gini']:.4f}  threshold=€{r['thresh']:,.0f}")

    high = flag_high_poverty(results)
    print(f"\nWaves above 20% AROP threshold: {high or 'none'}")

Overwriting debug_demo.py


---
### 🏋️ Exercise 1 — AROPE Indicator

The **AROPE** (At-Risk-of-Poverty OR Social Exclusion) is the EU 2030 headline poverty target. A person is AROPE if they meet **at least one** of:

1. At-risk-of-poverty (income < 60% of median)
2. Severely materially deprived (lacks ≥ 3 items from EU-SILC HS/HC list)
3. Living in a household with very low work intensity (adults work < 20% of capacity)

Add a function `arope_rate` to `silc_toolkit/indicators.py` that takes three boolean arrays (one per criterion) and returns the share meeting at least one.

Then write at least **two pytest tests** for it in `tests/test_indicators.py`.

In [ ]:
# 🏋️ Exercise 1 — Your solution here!
# Add arope_rate to indicators.py using %%writefile --append OR
# implement it directly and add the function here for testing.

def arope_rate(
    at_risk: list[bool],
    deprived: list[bool],
    low_work: list[bool],
) -> float:
    """
    AROPE rate: share meeting at least one of the three criteria.
    All lists must have the same length.
    """
    # TODO: implement
    pass


# TODO: write two tests


<details><summary>💡 Click for the solution</summary>

```python
def arope_rate(
    at_risk: list[bool],
    deprived: list[bool],
    low_work: list[bool],
) -> float:
    """AROPE rate: share meeting at least one of the three criteria."""
    n = len(at_risk)
    if n == 0:
        return 0.0
    assert len(deprived) == n == len(low_work), "All lists must have equal length"
    count = sum(
        1 for a, d, w in zip(at_risk, deprived, low_work)
        if a or d or w
    )
    return count / n


# Tests
def test_arope_none():
    n = 10
    assert arope_rate([False]*n, [False]*n, [False]*n) == 0.0

def test_arope_all():
    n = 10
    assert arope_rate([True]*n, [False]*n, [False]*n) == 1.0

def test_arope_union():
    # Only one criterion true per household — no overlap → sum of each rate
    at_risk = [True,  False, False, False]
    deprived = [False, True, False, False]
    low_work = [False, False, True, False]
    assert arope_rate(at_risk, deprived, low_work) == pytest.approx(0.75)
```
</details>

---
### 🏋️ Exercise 2 — Test Your Country

Using the participant countries' data:
1. Write a **parametrised test** `test_gini_range` that checks Gini is in [0.2, 0.5] for at least 4 countries
2. Write a **fixture** `be_incomes` for Belgium 2012 (copy the pattern from `lu_incomes` in conftest.py)
3. Add a test that Belgium's S80/S20 ratio is between 3 and 8

Run `pytest -v` to confirm all tests pass.

In [ ]:
# 🏋️ Exercise 2 — Your solution here!
# Extend conftest.py and test_indicators.py with %%writefile --append,
# or paste your new tests here for quick interactive testing.


<details><summary>💡 Click for the solution</summary>

**In conftest.py (add to existing file):**
```python
@pytest.fixture(scope="session")
def be_incomes(data_dir) -> list[float]:
    from silc_toolkit.loaders import load_incomes
    return load_incomes("BE", 2012, data_dir)
```

**In test_indicators.py:**
```python
@pytest.mark.parametrize("country", ["LU", "IE", "BE", "HU", "ES"])
def test_gini_range(data_dir, country):
    from silc_toolkit.loaders import load_incomes
    from silc_toolkit.indicators import gini_coefficient
    inc = load_incomes(country, 2012, data_dir)
    g = gini_coefficient(inc)
    assert 0.20 <= g <= 0.60, f"{country} Gini={g:.4f} out of plausible range"

def test_be_s80s20(be_incomes):
    from silc_toolkit.indicators import s80s20_ratio
    r = s80s20_ratio(be_incomes)
    assert 3.0 <= r <= 8.0, f"Belgium S80/S20={r:.2f} unexpected"
```
</details>

---
## 8 · Building the Distributable Package

So far we have used an **editable install** (`pip install -e .`) — perfect for development, but it only links to our source folder. To *ship* `silc_toolkit` to a colleague, a server, or [PyPI](https://pypi.org), we build a **distributable artifact**: a wheel (`.whl`) and a source distribution (`.tar.gz`).

This is what makes the package truly **installable** anywhere — the `[build-system]` and `[tool.hatch.build.targets.wheel]` blocks in our `pyproject.toml` tell the build backend (hatchling) exactly what to package.

| Artifact | File | Purpose |
|----------|------|---------|
| **Wheel** | `silc_toolkit-0.1.0-py3-none-any.whl` | Pre-built, fast to install — the modern default |
| **sdist** | `silc_toolkit-0.1.0.tar.gz` | Source archive — built on the target machine if no wheel fits |

The development workflow becomes a clean pipeline:

> this morning's code → editable dev install (§1.3) → **built, shippable wheel (this section)**


In [ ]:
# Build the wheel + sdist into dist/
# uv build is fast and uses the [build-system] backend (hatchling) from pyproject.toml.
# Falls back to `python -m build` if uv is unavailable.
import subprocess, sys, shutil

if shutil.which("uv"):
    cmd = ["uv", "build"]
else:
    cmd = [sys.executable, "-m", "build"]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-1500:])
if result.returncode == 0:
    print("\n✅ Build complete — artifacts written to dist/")
else:
    print("Build error:", result.stderr[-800:])


Build error: Building source distribution...
  × Failed to build `/content`
  ╰─▶ /content does not appear to be a Python project, as neither
      `pyproject.toml` nor `setup.py` are present in the directory



In [ ]:
# Inspect what was produced
dist = PROJECT_ROOT / "dist"
if dist.exists():
    for f in sorted(dist.iterdir()):
        print(f"  {f.name:45s} {f.stat().st_size / 1024:6.1f} KB")
else:
    print("No dist/ directory — run the build cell above first.")


### 8.1 Installing and Publishing

**An end-user** (not a developer) installs the built wheel directly — no source checkout, no editable link:

```bash
pip install dist/silc_toolkit-0.1.0-py3-none-any.whl
```

…or, on a clean machine, straight from the source distribution: `pip install dist/silc_toolkit-0.1.0.tar.gz`.

**Publishing to PyPI** so anyone can `pip install silc-toolkit` is one more step. We won't run it here (it requires an account and an API token), but the commands are:

```bash
# Classic toolchain
twine upload dist/*

# …or with uv
uv publish
```

> 💡 Always test against [TestPyPI](https://test.pypi.org) first: `twine upload --repository testpypi dist/*`. Remember to **bump the version** in `pyproject.toml` before every release — PyPI refuses to overwrite an existing version.


---
## 🗺️ Recap

| Concept | Key syntax | What it gave us |
|---------|-----------|------------------|
| **Package** | Directory + `__init__.py` | `from silc_toolkit import arop_rate` |
| **pyproject.toml** | `[project]` + `[tool.*]` | Reproducible install, tool config in one file |
| **Editable install** | `pip install -e .` | Live code changes, no reinstall needed |
| **Type hints** | `def f(x: list[float]) -> float` | Self-documentation + static analysis |
| **pytest fixture** | `@pytest.fixture` | Shared EU-SILC data loaded once per session |
| **parametrize** | `@pytest.mark.parametrize` | Same test across BE, ES, HU, IE, SE |
| **coverage** | `pytest --cov=silc_toolkit` | Find untested code paths |
| **black** | `black silc_toolkit/` | Zero-config auto-formatting |
| **ruff** | `ruff check silc_toolkit/` | Fast linting and style enforcement |
| **logging** | `logger.info("...")` | Structured runtime diagnostics |
| **pdb** | `breakpoint()` | Pause and inspect at any line |
| **build** | `uv build` → `dist/*.whl` | A shippable, `pip install`-able artifact |

---
## ⏭️ Up Next

**Notebook 03 — Advanced pandas & Polars** (tomorrow morning)

- Load all 10 years of Luxembourg data with chunking and optimised dtypes
- Multi-index DataFrames: `(country, year, region)`
- Rolling windows, `groupby`, `transform` for time-series AROP tracking
- Polars: re-implement AROP in 5 lines; benchmark 10× speedup
- Export to Parquet for efficient storage

🌙 *Great work today — see you at 9:00 tomorrow!*